# ✈️ TRIP.com Price Crawler — **v2** (speed-up)

Bản **v2** = parallel hotels + knobs tốc độ. Nếu không ổn → chạy lại [`../v1/run_trip.ipynb`](../v1/run_trip.ipynb) hoặc [`../run_trip.ipynb`](../run_trip.ipynb).

Chạy lần lượt: **① Cấu hình → ② Đọc input → ③ Crawl → ④ Xem kết quả**.

> ⚠️ Vẫn **BROWSER-only**. `SPEED_PROFILE="fast"` tăng concurrency (có thể bị block nhiều hơn).

**Output** tách riêng: `results/trip_v2/` (không đụng checkpoint của v1/root).

In [1]:
# ════════════════ ① CẤU HÌNH ════════════════

# ── Nguồn input: "gsheet" (online) hoặc "offline" (file trên máy) ──
INPUT_MODE = "gsheet"

# Dùng khi INPUT_MODE = "gsheet" (gid của tab được tự lấy từ URL)
GSHEET_URL = "https://docs.google.com/spreadsheets/d/1g_S06QeEAWnCTHYXGH0Nn4Mcb3FCT_-uIS1jUm4GGkw/edit?gid=607908359#gid=607908359"

# Dùng khi INPUT_MODE = "offline" — đường dẫn tuyệt đối, hoặc tương đối so với 31.crawl-tool
OFFLINE_FILE = "input/trip_hotels.csv"

# ── Tham số crawl ──
WEEKS      = 6      # số tuần cần crawl
MAX_HOTELS = 0      # 0 = crawl tất cả; đặt 5 để test nhanh 5 khách sạn đầu
SHARD      = ""     # "" = không chia; "1/2" = chạy phần 1 trong 2 phần

# ── Tốc độ (chỉ có ở v2) ──
# Trip hay soft-block khi mở NHIỀU HOTEL cùng lúc → mọi profile giữ hotels_parallel=1.
# Tăng tốc bằng weeks_parallel + giảm delay giữa hotels.
# "safe"   = giống v1
# "medium" = weeks×3, delay ngắn hơn (~ mục tiêu 7–10 phút/50)
# "fast"   = weeks×4, delay ngắn hơn nữa (vẫn 1 hotel/lần)
SPEED_PROFILE = "medium"

In [2]:
# ════════════════ ② ĐỌC INPUT ════════════════
import os, sys

_VERSION = 'v2'

def _find_roots(version):
    """Tìm PKG_ROOT (…/vN) + TOOL_ROOT (…/31.crawl-tool) kể cả khi cwd đang ở results/…"""
    cur = os.path.abspath("")
    seen = set()
    for _ in range(10):
        if cur in seen:
            break
        seen.add(cur)
        if os.path.basename(cur) == version and os.path.isdir(os.path.join(cur, "crawler")):
            return cur, os.path.dirname(cur)
        if os.path.isdir(os.path.join(cur, version, "crawler")):
            return os.path.join(cur, version), cur
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent
    raise RuntimeError(
        f"Không tìm thấy {version}/crawler (cwd={os.path.abspath('')!r}). "
        f"Mở notebook từ 31.crawl-tool/ hoặc {version}/, hoặc Restart Kernel rồi chạy lại từ cell ①.")

PKG_ROOT, TOOL_ROOT = _find_roots(_VERSION)
ROOT = TOOL_ROOT  # shared input/; results may be versioned below
os.chdir(TOOL_ROOT)  # ổn định cwd (tránh kẹt ở results/ sau cell crawl trước)

if PKG_ROOT not in sys.path:
    sys.path.insert(0, PKG_ROOT)

# Prefer this folder's crawler over a previously imported root crawler
if "crawler" in sys.modules:
    del sys.modules["crawler"]
    for k in list(sys.modules):
        if k.startswith("crawler."):
            del sys.modules[k]

import crawler
from crawler.hotels_io import read_hotels
print(f"📦 crawler {_VERSION} @ {PKG_ROOT} | version={getattr(crawler, '__version__', '?')}")
print(f"📂 TOOL_ROOT={TOOL_ROOT} | cwd={os.getcwd()}")

if INPUT_MODE == "gsheet":
    INPUT = GSHEET_URL
    print("📡 Input: Google Sheet online")
else:
    INPUT = OFFLINE_FILE if os.path.isabs(OFFLINE_FILE) else os.path.join(TOOL_ROOT, OFFLINE_FILE)
    assert os.path.exists(INPUT), f"Không tìm thấy file: {INPUT}"
    print(f"📁 Input: file offline — {INPUT}")

hotels = read_hotels(INPUT)
print(f"✅ Đọc được {len(hotels)} khách sạn. 5 dòng đầu:")
for name, url, room in hotels[:5]:
    print(f"   • {name} — {room}")


📦 crawler v2 @ /Users/hchinhtrung/Documents/GitHub/mvillage-email-template/31.crawl-tool/v2 | version=0.5.0-v2
📂 TOOL_ROOT=/Users/hchinhtrung/Documents/GitHub/mvillage-email-template/31.crawl-tool | cwd=/Users/hchinhtrung/Documents/GitHub/mvillage-email-template/31.crawl-tool
📡 Input: Google Sheet online
✅ Đọc được 55 khách sạn. 5 dòng đầu:
   • Halais Hotel — Superior Room
   • Minasi HanoiOi Hotel — Premier Double Bed Room
   • Muong Thanh Hanoi Centre Hotel — Superior King Room
   • Mercure Hanoi La Gare Hotel — Classic Double Bed Room With City View
   • REY Hotel Hanoi — Standard Twin Room


In [3]:
# (TÙY CHỌN) Tải Google Sheet về file offline — lần sau chỉ cần đổi INPUT_MODE = "offline"
import pandas as pd
from crawler.hotels_io import _gsheet_url

os.makedirs(os.path.join(ROOT, "input"), exist_ok=True)
dest = os.path.join(ROOT, "input", "trip_hotels.csv")
pd.read_csv(_gsheet_url(GSHEET_URL)).to_csv(dest, index=False, encoding="utf-8-sig")
print(f"💾 Đã lưu bản offline: {dest}")

💾 Đã lưu bản offline: /Users/hchinhtrung/Documents/GitHub/mvillage-email-template/31.crawl-tool/input/trip_hotels.csv


In [ ]:
# ════════════════ ③ CRAWL ════════════════
_prev_cwd = os.getcwd()
OUTDIR = os.path.join(ROOT, "results", "trip_v2")
os.makedirs(OUTDIR, exist_ok=True)
os.chdir(OUTDIR)                     # output (FINAL_*.csv, TEMP_*.csv) nằm ở đây

# Speed knobs (v2 only). Defaults match v1 when SPEED_PROFILE == "safe".
_PROFILES = {
    # Giống v1 — ổn định nhất
    "safe": dict(hotels_parallel=1, weeks_parallel=2,
                 between_hotels=(2.0, 5.0), nav_jitter=(0.3, 1.2),
                 intra_week_delay=(0.5, 1.5)),
    # 1 hotel × 3 weeks song song + delay ngắn — ít soft-block hơn multi-hotel
    "medium": dict(hotels_parallel=1, weeks_parallel=3,
                   between_hotels=(0.5, 1.2), nav_jitter=(0.2, 0.6),
                   intra_week_delay=(0.2, 0.6)),
    # 1 hotel × 4 weeks; nếu vẫn soft-block nhiều → hạ về medium/safe
    "fast": dict(hotels_parallel=1, weeks_parallel=4,
                 between_hotels=(0.3, 0.8), nav_jitter=(0.15, 0.4),
                 intra_week_delay=(0.15, 0.4)),
}
profile = _PROFILES.get(SPEED_PROFILE, _PROFILES["safe"])
print(f"⚡ SPEED_PROFILE={SPEED_PROFILE!r} → {profile}")

kwargs = dict(
    site="trip",
    input=INPUT,
    weeks=WEEKS,
    **profile,
)
if MAX_HOTELS:
    kwargs["max"] = MAX_HOTELS
if SHARD:
    kwargs["shard"] = SHARD

try:
    await crawler.arun(**kwargs)
finally:
    os.chdir(_prev_cwd)  # tránh kẹt cwd ở results/ khi chạy lại cell ②


⚡ SPEED_PROFILE='medium' → {'hotels_parallel': 1, 'weeks_parallel': 3, 'between_hotels': (0.5, 1.2), 'nav_jitter': (0.2, 0.6), 'intra_week_delay': (0.2, 0.6)}
📂 Resume: 40 rows from TEMP_trip.csv
🩺 env problems (/Users/hchinhtrung/Documents/GitHub/mvillage-email-template/31.crawl-tool/.venv/bin/python):
  ⚠️ camoufox browser binary missing → python -m camoufox fetch
🚀 TRIP crawl | 55 hotels × 6w | browser-only | engine=camoufox | hotels_parallel=1 | weeks_parallel=3 | W1=2026-07-27
✔️  1/55 Halais Hotel — complete, skip
✔️  2/55 Minasi HanoiOi Hotel — complete, skip
✔️  3/55 Muong Thanh Hanoi Centre Hotel — complete, skip
✔️  4/55 Mercure Hanoi La Gare Hotel — complete, skip
✔️  5/55 REY Hotel Hanoi — complete, skip
✔️  6/55 La Passion Premium Cau Go — complete, skip
✔️  7/55 Bespoke Trendy Hotel Hanoi (Formerly Hanoi La Siesta Hotel Trendy) — complete, skip
✔️  8/55 Nesta Hotel Hanoi — complete, skip
✔️  9/55 Silk Path Boutique Hanoi — complete, skip
✔️  10/55 La Siesta Premium Hang B

In [ ]:
# ════════════════ ④ XEM KẾT QUẢ ════════════════
import glob
import pandas as pd

OUTDIR = os.path.join(ROOT, "results", "trip_v2")
files = sorted(glob.glob(os.path.join(OUTDIR, "FINAL_*.csv")))
assert files, "Chưa có file FINAL nào — hãy chạy cell ③ trước."
latest = files[-1]
df = pd.read_csv(latest)
print(f"📄 {latest} — {len(df)} dòng")
df.head(20)
